# S2F Generated Graphs

Read graph artifacts produced by S2F and convert edge tables or sparse matrices into pandas DataFrames.

## Imports

In [1]:
from pathlib import Path
import configparser

import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

## Paths

In [2]:
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 's2f.conf').exists() else cwd.parent
if not (REPO_ROOT / 's2f.conf').exists():
    raise FileNotFoundError('Run this notebook from the S2F repo root or notebooks/ directory')

config = configparser.ConfigParser()
config.read(REPO_ROOT / 's2f.conf')
INSTALLATION_DIR = Path(config.get('directories', 'installation_directory')).expanduser()

# Change this alias to inspect another S2F run.
ALIAS = 'iAC560_genes'
MAX_EDGES = 100_000

OUTPUT_DIR = INSTALLATION_DIR / 'output' / ALIAS
GRAPH_DIR = INSTALLATION_DIR / 'graphs'
PROTEINS_PATH = OUTPUT_DIR / 'proteins.df'
COLLECTION_PATH = GRAPH_DIR / 'collection' / ALIAS
HOMOLOGY_NPZ = OUTPUT_DIR / 'homology.npz'
COMBINED_NPZ = GRAPH_DIR / 'combined' / f'{ALIAS}.npz'
COMBINED_EDGE_LIST = GRAPH_DIR / 'combined' / ALIAS

print(f'Repo root: {REPO_ROOT}')
print(f'Installation directory: {INSTALLATION_DIR}')
print(f'Alias: {ALIAS}')

Repo root: /home/marcelo_baez/paccanaro-lab/S2F
Installation directory: /media/marcelo_baez/HD_Disc1/.S2F
Alias: iAC560_genes


## Artifact Inventory

In [3]:
def artifact_row(name, path):
    path = Path(path)
    exists = path.exists()
    return {
        'artifact': name,
        'path': str(path),
        'exists': exists,
        'size_mb': round(path.stat().st_size / 1024**2, 3) if exists else np.nan,
    }

artifacts = pd.DataFrame([
    artifact_row('protein index', PROTEINS_PATH),
    artifact_row('graph collection pickle', COLLECTION_PATH),
    artifact_row('homology sparse matrix', HOMOLOGY_NPZ),
    artifact_row('combined sparse matrix', COMBINED_NPZ),
    artifact_row('combined edge list', COMBINED_EDGE_LIST),
])
display(artifacts)

,artifact,path,exists,size_mb
0,protein index,/media/marcelo_baez/HD_Disc1/.S2F/output/iAC56...,True,0.013
1,graph collection pickle,/media/marcelo_baez/HD_Disc1/.S2F/graphs/colle...,True,0.733
2,homology sparse matrix,/media/marcelo_baez/HD_Disc1/.S2F/output/iAC56...,False,NaN
3,combined sparse matrix,/media/marcelo_baez/HD_Disc1/.S2F/graphs/combi...,False,NaN
4,combined edge list,/media/marcelo_baez/HD_Disc1/.S2F/graphs/combi...,False,NaN


## Protein Index

In [ ]:
proteins = pd.read_pickle(PROTEINS_PATH)
if 'protein idx' not in proteins.columns:
    raise ValueError(f'{PROTEINS_PATH} does not contain a protein idx column')

protein_lookup = proteins.reset_index()
protein_id_col = proteins.index.name or protein_lookup.columns[0]
protein_lookup = protein_lookup.rename(columns={protein_id_col: 'protein id'})
protein_lookup = protein_lookup[['protein id', 'protein idx']].sort_values('protein idx')
protein_ids_by_idx = protein_lookup.set_index('protein idx')['protein id']

print(f'Loaded {len(proteins):,} proteins')
display(protein_lookup.head())

## Graph Collection

In [4]:
collection = pd.read_pickle(COLLECTION_PATH)
score_columns = [
    col for col in collection.columns
    if col not in {'query1', 'query2', 'max_evalue'} and pd.api.types.is_numeric_dtype(collection[col])
]

print(f'Loaded collection with {len(collection):,} transferred edges and {len(score_columns)} score columns')
display(collection.head())

collection_summary = pd.DataFrame([
    {
        'graph': col,
        'positive_edges': int((collection[col] > 0).sum()),
        'min_positive': collection.loc[collection[col] > 0, col].min(),
        'max': collection[col].max(),
        'mean_positive': collection.loc[collection[col] > 0, col].mean(),
    }
    for col in score_columns
]).sort_values('positive_edges', ascending=False)
display(collection_summary)

Loaded collection with 11,974 transferred edges and 5 score columns


,neighborhood,experiments,coexpression,textmining,database,query1,query2,max_evalue
0,696,0,0,0,900,LmjF.05.0980,LmjF.27.0740,1.740000e-72
1,696,0,0,0,900,LmjF.05.0980,LmjF.27.0740,1.740000e-72
2,0,0,0,0,900,LmjF.24.1630,LmjF.25.2140,2.730000e-124
3,0,0,0,0,900,LmjF.24.1630,LmjF.25.2140,2.730000e-124
4,0,0,0,0,900,LmjF.24.0320,LmjF.24.1630,0.000000e+00


,graph,positive_edges,min_positive,max,mean_positive
4,database,6384,360,900,698.956767
3,textmining,4652,46,996,340.343078
2,coexpression,2328,41,999,95.170962
0,neighborhood,384,148,773,509.770833
1,experiments,106,59,981,324.226415


In [5]:
if score_columns:
    collection_edges = collection.melt(
        id_vars=['query1', 'query2', 'max_evalue'],
        value_vars=score_columns,
        var_name='graph',
        value_name='score',
    )
    collection_edges = collection_edges[collection_edges['score'] > 0]
    collection_edges = collection_edges.sort_values(['score', 'graph'], ascending=[False, True]).reset_index(drop=True)
else:
    collection_edges = pd.DataFrame(columns=['query1', 'query2', 'max_evalue', 'graph', 'score'])

print(f'Positive scored collection edges: {len(collection_edges):,}')
display(collection_edges.head(20))

Positive scored collection edges: 13,854


,query1,query2,max_evalue,graph,score
0,LmjF.28.2430,LmjF.34.3670,0.000000e+00,coexpression,999
1,LmjF.28.2430,LmjF.34.3670,0.000000e+00,coexpression,999
2,LmjF.05.0980,LmjF.27.0740,1.930000e-68,textmining,996
3,LmjF.05.0980,LmjF.27.0740,1.930000e-68,textmining,996
4,LmjF.27.2050,LmjF.28.0890,3.900000e-147,textmining,993
5,LmjF.27.2050,LmjF.28.0890,3.900000e-147,textmining,993
6,LmjF.28.2430,LmjF.34.3670,0.000000e+00,experiments,981
7,LmjF.28.2430,LmjF.34.3670,0.000000e+00,experiments,981
8,LmjF.05.0980,LmjF.27.0740,1.930000e-68,experiments,954
9,LmjF.05.0980,LmjF.27.0740,1.930000e-68,experiments,954


In [8]:
EXPORT_DIR = REPO_ROOT / 'notebooks' / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

collection_edges_path = EXPORT_DIR / 'interolog_transferred_edges.csv'
collection_edges.to_csv(collection_edges_path, index=False)

print(f'Saved collection_edges to: {collection_edges_path.resolve()}')
print(f'File exists: {collection_edges_path.exists()}')
print(f'File size: {collection_edges_path.stat().st_size / 1024**2:.3f} MB')

## Sparse Matrix Helpers

In [ ]:
def protein_id_map(proteins_df):
    lookup = proteins_df.reset_index()
    protein_id_col = proteins_df.index.name or lookup.columns[0]
    lookup = lookup.rename(columns={protein_id_col: 'protein id'})
    return lookup.set_index('protein idx')['protein id']


def sparse_graph_to_edges(path, proteins_df, max_edges=100_000, upper_triangle_only=True):
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(path)

    graph = sparse.load_npz(path).tocoo(copy=False)
    row = graph.row
    col = graph.col
    data = graph.data
    limit = graph.nnz if max_edges is None else min(int(max_edges), graph.nnz)
    selected = []
    remaining = limit
    chunk_size = max(1_000_000, limit * 5 if limit else 1_000_000)

    for start in range(0, graph.nnz, chunk_size):
        stop = min(start + chunk_size, graph.nnz)
        if upper_triangle_only:
            local = np.flatnonzero(row[start:stop] < col[start:stop])
        else:
            local = np.arange(stop - start)
        if local.size == 0:
            continue
        take = local[:remaining] + start
        selected.append(take)
        remaining -= len(take)
        if remaining <= 0:
            break

    if selected:
        positions = np.concatenate(selected)
    else:
        positions = np.array([], dtype=np.intp)

    edges = pd.DataFrame({
        'protein1_idx': row[positions],
        'protein2_idx': col[positions],
        'score': data[positions],
    })
    ids_by_idx = protein_id_map(proteins_df)
    edges['protein1'] = edges['protein1_idx'].map(ids_by_idx)
    edges['protein2'] = edges['protein2_idx'].map(ids_by_idx)
    edges = edges[['protein1', 'protein2', 'score', 'protein1_idx', 'protein2_idx']]
    return graph, edges


print(f'Helper ready. Sparse loaders will convert at most {MAX_EDGES:,} edges to pandas by default.')

## Homology Graph

In [ ]:
if HOMOLOGY_NPZ.exists():
    homology_graph, homology_edges = sparse_graph_to_edges(HOMOLOGY_NPZ, proteins, max_edges=MAX_EDGES)
    print(f'Homology matrix: shape={homology_graph.shape}, nnz={homology_graph.nnz:,}')
    print(f'Showing {len(homology_edges):,} upper-triangle edges')
    display(homology_edges.head(20))
else:
    homology_graph = None
    homology_edges = pd.DataFrame(columns=['protein1', 'protein2', 'score', 'protein1_idx', 'protein2_idx'])
    print(f'Missing homology graph: {HOMOLOGY_NPZ}')

## Combined Graph

In [ ]:
if COMBINED_NPZ.exists():
    combined_graph, combined_edges = sparse_graph_to_edges(COMBINED_NPZ, proteins, max_edges=MAX_EDGES)
    print(f'Combined matrix: shape={combined_graph.shape}, nnz={combined_graph.nnz:,}')
    print(f'Showing {len(combined_edges):,} upper-triangle edges')
    display(combined_edges.head(20))
elif COMBINED_EDGE_LIST.exists():
    combined_graph = None
    combined_edges = pd.read_csv(COMBINED_EDGE_LIST, sep='\t', names=['protein1', 'protein2', 'score'])
    print(f'Loaded combined edge list with {len(combined_edges):,} edges')
    display(combined_edges.head(20))
else:
    combined_graph = None
    combined_edges = pd.DataFrame(columns=['protein1', 'protein2', 'score'])
    print(f'Missing combined sparse matrix: {COMBINED_NPZ}')
    print(f'Missing combined edge list: {COMBINED_EDGE_LIST}')